**all requirements**

In [62]:
# !pip install torch
# !pip install hf_xet
# !pip install skillNer
# !python -m spacy download en_core_web_lg

**imports**

In [ ]:
"""
CV Parser - Extract and Clean Resume Data from PDF
This script extracts text from CV PDFs, cleans it, and extracts key information
including name, email, phone, education, experience, and skills.
"""

import re
import os
import spacy
from pdfminer.high_level import extract_text
from pdfminer.layout import LAParams
import string

# Download required model: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

**NER approach**    

In [64]:
class CVParser:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.raw_text = ""
        self.cleaned_text = ""
        self.extracted_data = {}
        
        # Common skills database (expandable)
        self.skills_database = {
            'programming': ['python', 'java', 'javascript', 'c++', 'c#', 'ruby', 'php', 
                          'swift', 'kotlin', 'go', 'rust', 'typescript', 'sql', 'r'],
            'web': ['html', 'css', 'react', 'angular', 'vue', 'nodejs', 'django', 
                   'flask', 'spring', 'asp.net', 'jquery', 'bootstrap', 'tailwind'],
            'data': ['pandas', 'numpy', 'matplotlib', 'scikit-learn', 'tensorflow', 
                    'pytorch', 'keras', 'spark', 'hadoop', 'tableau', 'power bi'],
            'cloud': ['aws', 'azure', 'gcp', 'docker', 'kubernetes', 'jenkins', 
                     'terraform', 'ansible', 'ci/cd', 'devops'],
            'databases': ['mysql', 'postgresql', 'mongodb', 'oracle', 'redis', 
                         'elasticsearch', 'cassandra', 'dynamodb'],
            'tools': ['git', 'jira', 'confluence', 'slack', 'agile', 'scrum', 
                     'linux', 'windows', 'macos', 'vs code', 'intellij'],
            'soft': ['leadership', 'communication', 'teamwork', 'problem solving', 
                    'analytical', 'critical thinking', 'project management']
        }
        
    def extract_text_from_pdf(self):
        """Extract text from PDF file"""
        try:
            # LAParams helps preserve layout structure
            self.raw_text = extract_text(self.pdf_path)
            print("✓ Text extracted from PDF successfully")
            return self.raw_text
        except Exception as e:
            print(f"✗ Error extracting text: {e}")
            return ""
    
    def clean_text(self, text):
        """Clean and preprocess text"""
        if not text:
            return ""
        
        # Remove bullet points and special symbols
        bullets = ['•', '●', '○', '■', '□', '▪', '▫', '–', '—', '→', '►', '✓', '✔']
        for bullet in bullets:
            text = text.replace(bullet, '')
        
        # Remove extra whitespace and newlines
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub('#\S+', '', text)  # remove hashtags
        text = re.sub('@\S+', '  ', text)  # remove mentions
        text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)  # remove punctuations
        text = re.sub(r'[^\x00-\x7f]',r' ', text) 
        text = re.sub('\s+', ' ', text)  # remove extra whitespace
        # Keep only alphanumeric, @, +, -, ., and basic punctuation
        text = re.sub(r'[^a-zA-Z0-9@+\-.,:()\n\s]', '', text)
        
        # Remove standalone numbers (likely from formatting)
        text = re.sub(r'\b\d{1,2}\b', '', text)
        
        self.cleaned_text = text.strip()
        print("✓ Text cleaned successfully")
        return self.cleaned_text
    
    def lemmatize_text(self, text):
        """Lemmatize text using spaCy"""
        doc = nlp(text)
        
        # Lemmatize while preserving important entities
        lemmatized_tokens = []
        for token in doc:
            # Skip stop words except important ones
            if token.is_stop and token.text.lower() not in ['years', 'year', 'experience']:
                continue
            # Keep entities as-is, lemmatize others
            if token.ent_type_:
                lemmatized_tokens.append(token.text)
            else:
                lemmatized_tokens.append(token.lemma_)
        
        lemmatized_text = ' '.join(lemmatized_tokens)
        print("✓ Text lemmatized successfully")
        return lemmatized_text
    
    def extract_name(self, text):
        """Extract name from CV (usually at the top)"""
        lines = text.split('\n')
        
        # Try first few lines for name
        for line in lines[:5]:
            line = line.strip()
            if len(line) > 3 and len(line) < 50:
                # Check if line contains likely name pattern
                doc = nlp(line)
                for ent in doc.ents:
                    if ent.label_ == "PERSON":
                        return ent.text
                
                # Fallback: check if it's mostly alphabetic
                if sum(c.isalpha() or c.isspace() for c in line) / len(line) > 0.8:
                    return line
        
        return "Not Found"
    
    def extract_email(self, text):
        """Extract email address"""
        email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        emails = re.findall(email_pattern, text)
        return emails[0] if emails else "Not Found"
    
    def extract_phone(self, text):
        """Extract phone number"""
        # Multiple phone patterns
        patterns = [
            r'\+?\d{1,3}[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',  # +1-234-567-8900
            r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',  # (234) 567-8900
            r'\d{10,}',  # 2345678900
        ]
        
        for pattern in patterns:
            phones = re.findall(pattern, text)
            if phones:
                return phones[0]
        
        return "Not Found"
    
    def extract_education(self, text):
        """Extract education information"""
        education = []
        
        # Common degree patterns
        degree_patterns = [
            r'(Bachelor|Master|PhD|Ph\.D|MBA|B\.S|M\.S|B\.A|M\.A|B\.Tech|M\.Tech|BE|ME|BSc|MSc)',
            r'(Undergraduate|Graduate|Doctorate|Associates|Diploma)'
        ]
        
        lines = text.split('\n')
        in_education_section = False
        
        for i, line in enumerate(lines):
            # Check if we're in education section
            if re.search(r'\b(education|academic|qualification|degree)\b', line, re.IGNORECASE):
                in_education_section = True
                continue
            
            # Exit education section if we hit another major section
            if in_education_section and re.search(r'\b(experience|skills|projects|work)\b', line, re.IGNORECASE):
                in_education_section = False
            
            # Look for degrees
            for pattern in degree_patterns:
                if re.search(pattern, line, re.IGNORECASE):
                    # Extract year if present
                    year_match = re.search(r'\b(19|20)\d{2}\b', line)
                    year = year_match.group(0) if year_match else ""
                    
                    education.append({
                        'degree': line.strip(),
                        'year': year
                    })
        
        return education if education else [{"degree": "Not Found", "year": ""}]
    
    def extract_experience_years(self, text):
        """Extract years of experience"""
        # Patterns for experience
        patterns = [
            r'(\d+)\+?\s*years?\s+(?:of\s+)?experience',
            r'experience[:\s]+(\d+)\+?\s*years?',
            r'(\d+)\+?\s*yrs?\s+(?:of\s+)?experience',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return f"{match.group(1)}+ years"
        
        # Count experience entries
        experience_count = len(re.findall(r'\b(19|20)\d{2}\s*-\s*(?:(19|20)\d{2}|present|current)', text, re.IGNORECASE))
        if experience_count > 0:
            return f"~{experience_count} positions listed"
        
        return "Not specified"
    
    def extract_experience(self, text):
        """Extract work experience details"""
        experience = []
        lines = text.split('\n')
        in_experience_section = False
        current_exp = {}
        
        for line in lines:
            # Check if we're in experience section
            if re.search(r'\b(experience|employment|work history|professional)\b', line, re.IGNORECASE):
                in_experience_section = True
                continue
            
            # Exit if we hit another section
            if in_experience_section and re.search(r'\b(education|skills|projects|certification)\b', line, re.IGNORECASE):
                if current_exp:
                    experience.append(current_exp)
                break
            
            if in_experience_section:
                # Look for date ranges (experience period)
                date_match = re.search(r'(19|20)\d{2}\s*-\s*(?:(19|20)\d{2}|present|current)', line, re.IGNORECASE)
                if date_match:
                    if current_exp:
                        experience.append(current_exp)
                    current_exp = {'period': date_match.group(0), 'role': ''}
                
                # Look for job titles/roles
                elif current_exp and not current_exp.get('role') and len(line.strip()) > 5:
                    current_exp['role'] = line.strip()
        
        if current_exp:
            experience.append(current_exp)
        
        return experience if experience else [{"role": "Not Found", "period": ""}]
    
    def extract_skills(self, text):
        """Extract skills using multiple approaches"""
        text_lower = text.lower()
        found_skills = set()
        
        # Approach 1: Match against skills database
        for category, skills_list in self.skills_database.items():
            for skill in skills_list:
                # Word boundary matching
                pattern = r'\b' + re.escape(skill) + r'\b'
                if re.search(pattern, text_lower):
                    found_skills.add(skill.title())
        
        # Approach 2: Find skills section and extract
        lines = text.split('\n')
        in_skills_section = False
        
        for line in lines:
            # Detect skills section
            if re.search(r'\b(skills|technical skills|technologies|competencies|expertise)\b', line, re.IGNORECASE):
                in_skills_section = True
                continue
            
            # Exit skills section
            if in_skills_section and re.search(r'\b(experience|education|projects|certification)\b', line, re.IGNORECASE):
                break
            
            # Extract skills from skills section
            if in_skills_section:
                # Split by common delimiters
                skills_in_line = re.split(r'[,;|•]', line)
                for skill in skills_in_line:
                    skill = skill.strip()
                    if 2 < len(skill) < 30:  # Reasonable skill name length
                        found_skills.add(skill.title())
        
        # Approach 3: Use NLP to find technical terms
        doc = nlp(text)
        for chunk in doc.noun_chunks:
            chunk_text = chunk.text.lower()
            # Check if it's a potential skill (short phrases, contains keywords)
            if len(chunk_text.split()) <= 3 and any(tech in chunk_text for tech in ['data', 'web', 'software', 'system', 'development']):
                found_skills.add(chunk.text.title())
        
        return sorted(list(found_skills)) if found_skills else ["Not Found"]
    
    def parse(self):
        """Main parsing function"""
        print("\n" + "="*50)
        print("Starting CV Parsing...")
        print("="*50 + "\n")
        
        # Step 1: Extract text
        self.extract_text_from_pdf()
        
        # Step 2: Clean text
        cleaned = self.clean_text(self.raw_text)
        
        # Step 3: Lemmatize
        lemmatized = self.lemmatize_text(cleaned)
        
        # Step 4: Extract information
        print("\n" + "-"*50)
        print("Extracting Information...")
        print("-"*50 + "\n")
        
        self.extracted_data = {
            'name': self.extract_name(self.raw_text),
            'email': self.extract_email(self.raw_text),
            'phone': self.extract_phone(self.raw_text),
            'education': self.extract_education(self.raw_text),
            'years_of_experience': self.extract_experience_years(self.raw_text),
            'experience': self.extract_experience(self.raw_text),
            'skills': self.extract_skills(lemmatized)
        }
        
        return self.extracted_data
    
    def display_results(self):
        """Display extracted information in a formatted way"""
        print("\n" + "="*50)
        print("EXTRACTED CV INFORMATION")
        print("="*50 + "\n")
        
        print(f"📌 Name: {self.extracted_data['name']}")
        print(f"📧 Email: {self.extracted_data['email']}")
        print(f"📱 Phone: {self.extracted_data['phone']}")
        print(f"⏳ Experience: {self.extracted_data['years_of_experience']}")
        
        print(f"\n🎓 Education:")
        for edu in self.extracted_data['education']:
            print(f"   • {edu['degree']}" + (f" ({edu['year']})" if edu['year'] else ""))
        
        print(f"\n💼 Work Experience:")
        for exp in self.extracted_data['experience']:
            print(f"   • {exp['role']}" + (f" | {exp['period']}" if exp['period'] else ""))
        
        print(f"\n🛠 Skills ({len(self.extracted_data['skills'])} found):")
        skills_text = ', '.join(self.extracted_data['skills'][:20])  # Show first 20
        print(f"   {skills_text}")
        if len(self.extracted_data['skills']) > 20:
            print(f"   ... and {len(self.extracted_data['skills']) - 20} more")
        
        print("\n" + "="*50 + "\n")



In [65]:
# ============================================================================
# USAGE EXAMPLE
# ============================================================================

if __name__ == "__main__":
    # Replace with your CV file path
    cv_path = "./Eman_Ashraf_CV.pdf"

    if not os.path.exists(cv_path):
        print(f"❌ File not found: {cv_path}")
    else:
        parser = CVParser(cv_path)
        data = parser.parse()
        parser.display_results()

    
    # Access individual fields
    print("Accessing individual fields:")
    print(f"Name: {data['name']}")
    print(f"Skills: {data['skills']}")


Starting CV Parsing...

✓ Text extracted from PDF successfully
✓ Text cleaned successfully
✓ Text lemmatized successfully

--------------------------------------------------
Extracting Information...
--------------------------------------------------


EXTRACTED CV INFORMATION

📌 Name: EMAN ASHRAF | Software Engineer
📧 Email: emanashraf9090ea@gmail.com
📱 Phone: +20 0111629923
⏳ Experience: Not specified

🎓 Education:
   • About Me
   • Bachelor of Science in Computer Science from Ain Shams University with hands-on experience in Python, Java,
   • Bachelor of Science in Computer Science
   • October 2022 – August 2026 (2022)
   • Beyond Earth – Cosmic Journey | FastAPI,Python,JS,HTML,CSS
   • september 2025 – October 2025 (2025)
   • • Developed the backend of the Beyond Earth – Cosmic Journey project using FastAPI, enabling seamless data flow
   • between the interactive 3D frontend and scientific exoplanet datasets.
   • • Implemented secure OTP-based email verification to enhance au

**Amjad/awad for skill extraction**

In [66]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

cv_path = "./Eman_Ashraf_CV.pdf"

parser = CVParser(cv_path)

cv_text=parser.extract_text_from_pdf()

cleaned = parser.clean_text(cv_text)

lemmatized = parser.lemmatize_text(cleaned)

import time

start = time.time()
from huggingface_hub import snapshot_download
import spacy

# Download the model from the Hub
model_path = snapshot_download("amjad-awad/skill-extractor", repo_type="model")

# Load the model with spaCy
nlp = spacy.load(model_path)

# Example usage
doc = nlp(lemmatized)

# Extract skill entities
skills = [ent.text for ent in doc.ents if "SKILLS" in ent.label_]
print(skills)

end = time.time()
print("⏱️ Execution time:", end - start, "seconds")


✓ Text extracted from PDF successfully
✓ Text cleaned successfully
✓ Text lemmatized successfully


Fetching 15 files: 100%|██████████| 15/15 [00:00<?, ?it/s]


['GitHub', 'Python', 'Java', 'C', 'C', 'machine learning', 'Python', 'JS', 'PHP', 'MySQL', 'mysql', 'HTML', 'css', 'JS', 'HTML', 'CSS', 'JavaScript', 'code', 'C', 'graph', 'Windows', 'Python Machine Learning', 'C', 'Python', 'Java', 'C', 'C', 'JavaScript', 'C', 'TypeScript', 'Scala', 'Git', 'GitHub', 'MySQL', 'HTML', 'computer vision', 'deep learning', 'object detection']
⏱️ Execution time: 6.5166871547698975 seconds


**skillNER for skill extraction**

In [67]:
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

nlp = spacy.load("en_core_web_sm")

# Pass the PhraseMatcher class, not an instance
skill_extractor = SkillExtractor(
    nlp,
    SKILL_DB,
    PhraseMatcher  # ✅ Pass the class, not an instance
)

print("✅ SkillExtractor loaded")

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
✅ SkillExtractor loaded


In [68]:
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

nlp = spacy.load("en_core_web_lg")


import time

start = time.time()

annotations = skill_extractor.annotate(lemmatized)

skills_list = []

# full matches
for item in annotations['results'].get('full_matches', []):
    skills_list.append(item['doc_node_value'])

# ngram scored
for item in annotations['results'].get('ngram_scored', []):
    skills_list.append(item['doc_node_value'])

# remove duplicates
skills_list = list(set(skills_list))

print("Skills found:")
for skill in skills_list:
    print("-", skill)

 

end = time.time()
print("⏱️ Execution time:", end - start, "seconds")

Skills found:
- javascript
- html
- bootstrap
- project implement
- computer vision
- access control
- image processing
- software engineer
- track
- git
- data integrity
- programming concept
- scale
- github
- analysis user
- java
- integration
- science
- object detection
- c
- management system
- priority queue
- css
- short path
- operating system
- typescript
- node
- dataset
- computer science
- mysql
- machine learn
- languages arabic
- sales
- engineer feature
- dynamic memory allocation
- HTML
- data structure
- CSS
- authorization
- memory management
- data
- window form
- prediction
- regression model
- model predict
- object orient programming
- source
- algorithms
- OOP
- system architecture
- science computer
- backend
- read
- c c
- english
- laravel
- end end
- code structure
- scala
- web application
- programming languages
- python
- deep learning
- php
⏱️ Execution time: 7.8651041984558105 seconds


**yashpwr/resume-ner-bert-v2**

In [69]:
cv_path = "./12334140.pdf"

parser = CVParser(cv_path)

cv_text=parser.extract_text_from_pdf()

cleaned = parser.clean_text(cv_text)

lemmatized = parser.lemmatize_text(cleaned)

import time

start = time.time()

from transformers import pipeline

ner = pipeline(
  "token-classification",
  model="yashpwr/resume-ner-bert-v2",
  aggregation_strategy="simple"
)

entities = ner(lemmatized)
# Assuming `entities` from the ner pipeline
high_conf_degrees = [
    e['word'] 
    for e in entities 
    if e['entity_group'] == 'Degree' and e['score'] > 0.7
]

# Remove very short tokens (<3 characters) which are often noise
cleaned_degrees = [d for d in high_conf_degrees if len(d.split()) > 2]

for d in cleaned_degrees:
    print(d)

end = time.time()
print("⏱️ Execution time:", end - start, "seconds")

✓ Text extracted from PDF successfully
✓ Text cleaned successfully
✓ Text lemmatized successfully


Device set to use cpu


related experience excellent capability appreciate consideration job open skill ideal match position requirement responsible evening operation Student Center facility include
manage registration solve customer problem deal risk
management emergency enforcement department policy assist hire training
management staff Coordinate statistic inventory experience supervision student staff strong interpersonal skill prefer Valid Minnesota driver s license good driving record ability travel different site require experience
management Qualifications Register student course design
manage program software solve customer problem enforce department policy serve contact
manage supply inventory order Minnesota driver s license N
management excellent interpersonal communication skill appreciate take time review credential experience
copy letter Typed information TECHNOLOGY technician TIER TECHNICAL support help DESK TECHNICIAN Experienced knowledgeable Information Technology Professional seek contribu

**dslim/bert-base-NER**

In [70]:
from huggingface_hub import snapshot_download
import spacy

cv_path = "./Eman_Ashraf_CV.pdf"

parser = CVParser(cv_path)

cv_text=parser.extract_text_from_pdf()

cleaned = parser.clean_text(cv_text)

lemmatized = parser.lemmatize_text(cleaned)
import time

start = time.time()

from transformers import pipeline

ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")


entities = ner_pipeline(lemmatized)
print(entities)

end = time.time()
print("⏱️ Execution time:", end - start, "seconds")

✓ Text extracted from PDF successfully
✓ Text cleaned successfully
✓ Text lemmatized successfully


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'entity_group': 'ORG', 'score': 0.8521662, 'word': 'EMA', 'start': 0, 'end': 3}, {'entity_group': 'ORG', 'score': 0.82698613, 'word': 'AS', 'start': 5, 'end': 7}, {'entity_group': 'LOC', 'score': 0.9486724, 'word': 'Cairo Egypt', 'start': 61, 'end': 72}, {'entity_group': 'ORG', 'score': 0.68043417, 'word': '##H', 'start': 85, 'end': 86}, {'entity_group': 'ORG', 'score': 0.7037495, 'word': 'ICPCI', 'start': 100, 'end': 105}, {'entity_group': 'ORG', 'score': 0.80487084, 'word': 'Science Computer Science', 'start': 116, 'end': 140}, {'entity_group': 'ORG', 'score': 0.9959124, 'word': 'Ain Shams University', 'start': 141, 'end': 161}, {'entity_group': 'MISC', 'score': 0.80647635, 'word': 'Python Java C', 'start': 178, 'end': 191}, {'entity_group': 'ORG', 'score': 0.9068085, 'word': 'Education Ain Shams University Cairo Egypt', 'start': 340, 'end': 382}, {'entity_group': 'ORG', 'score': 0.7650647, 'word': 'Science Computer Science Projects', 'start': 392, 'end': 425}, {'entity_group': 'OR

**AventIQ-AI/Resume-Parsing-NER-AI-Model**

In [71]:
import time

start = time.time()
from transformers import pipeline


ner_pipeline = pipeline(
    "token-classification",
    model="AventIQ-AI/Resume-Parsing-NER-AI-Model",
    aggregation_strategy="simple"
)

entities = ner_pipeline(lemmatized)

print(entities)
end = time.time()
print("⏱️ Execution time:", end - start, "seconds")

Device set to use cpu


[{'entity_group': 'EMAIL', 'score': 0.43236834, 'word': 'E', 'start': 0, 'end': 1}, {'entity_group': 'PHONE', 'score': 0.9380317, 'word': 'Cairo', 'start': 61, 'end': 66}, {'entity_group': 'PHONE', 'score': 0.8198493, 'word': 'Egypt', 'start': 67, 'end': 72}, {'entity_group': 'EMAIL', 'score': 0.75830543, 'word': 'Ain Shams University', 'start': 141, 'end': 161}, {'entity_group': 'EDUCATION', 'score': 0.5952399, 'word': 'C', 'start': 190, 'end': 191}, {'entity_group': 'EMAIL', 'score': 0.7696143, 'word': 'Ain', 'start': 350, 'end': 353}, {'entity_group': 'PHONE', 'score': 0.6003589, 'word': 'Shams', 'start': 354, 'end': 359}, {'entity_group': 'EMAIL', 'score': 0.543922, 'word': 'University', 'start': 360, 'end': 370}, {'entity_group': 'PHONE', 'score': 0.9121419, 'word': 'Cairo', 'start': 371, 'end': 376}, {'entity_group': 'PHONE', 'score': 0.94502056, 'word': 'Egypt', 'start': 377, 'end': 382}, {'entity_group': 'EDUCATION', 'score': 0.7040284, 'word': 'Cosmic Journey', 'start': 457, '

**ihk/skillner**

In [72]:
import time
import spacy
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

start = time.time()

# Load spaCy for sentence splitting
nlp = spacy.load("en_core_web_sm")

model_name = "ihk/skillner"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

skill_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# Split text into sentences
doc = nlp(lemmatized)
sentences = [sent.text for sent in doc.sents]

# Process each sentence individually
all_results = []
for sentence in sentences:
    if len(tokenizer.encode(sentence)) <= 512:  # Check if sentence fits
        results = skill_pipeline(sentence)
        all_results.extend(results)

skills = [e['word'] for e in all_results if e['entity_group'].endswith("SKILL") and e['score'] > 0.5]
print("Extracted skills:", skills)
end = time.time()
print("⏱️ Execution time:", end - start, "seconds")

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Extracted skills: ['G', 'Code', 'Python Java C PHP', 'system architecture machine learning', 'OTP base', 'verification', 'authentication', '##vel', '##QL', 'Laravel base', 'read update delete functionality', 'user authentication role base', 'access control', '##ation', 'integrity', '##ive', 'Machine Learning', 'regression classification model', 'game sale', '##par', 'Java', 'JavaScript', 'TypeScript', 'Scala', 'Object Oriented', 'OOP', 'Data Structures', 'Too', 'Git', 'GitHub', 'MySQL', 'HTML CSS', 'Fast', 'Laravel', '##p', 'learning model image processing', 'detection recognition']
⏱️ Execution time: 5.999412298202515 seconds
